In [ ]:
import sys, subprocess, pkgutil

def ensure(pip_name, import_name=None):
    import_name = import_name or pip_name
    if pkgutil.find_loader(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pip_name])

# Обычно в Colab это уже есть, но пусть будет надёжно:
ensure("tqdm")
ensure("seaborn")
ensure("pandas")
ensure("scikit-learn")
# cv2 обычно есть; если вдруг нет:
if pkgutil.find_loader("cv2") is None:
    ensure("opencv-python-headless", "cv2")

/tmp/ipykernel_1229/4277555641.py:5: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if pkgutil.find_loader(import_name) is None:
/tmp/ipykernel_1229/4277555641.py:14: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if pkgutil.find_loader("cv2") is None:


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, zipfile, shutil, re
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/CMC_1/public_tests_dis.zip")
EXTRACT_DIR = Path("/content/datasets/public_tests_dis")   # сюда распакован архив (или будет распакован)
TRAIN_DIR = Path("/content/train_dataset")             # сюда соберём "единый train_dataset" как ожидает ноут

assert ZIP_PATH.exists(), f"Не найден архив: {ZIP_PATH}"

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# 1) Проверяем, что распаковка уже есть (есть папки *_test_file_input)
case_dirs = sorted([p for p in EXTRACT_DIR.iterdir()
                    if p.is_dir() and re.match(r"^\d+_test_file_input$", p.name)])

# 2) Если нет — распаковываем
if len(case_dirs) == 0:
    print("📦 Не вижу распакованных *_test_file_input — распаковываю zip...")
    shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)

    case_dirs = sorted([p for p in EXTRACT_DIR.iterdir()
                        if p.is_dir() and re.match(r"^\d+_test_file_input$", p.name)])

assert case_dirs, f"После распаковки всё равно не нашёл *_test_file_input в {EXTRACT_DIR}"

print("✅ Найдены кейсы:", [p.name for p in case_dirs])

# 3) Пересобираем /content/train_dataset (как ожидает ноут): video/NN.mp4 и gt/NN.json
if TRAIN_DIR.exists() or TRAIN_DIR.is_symlink():
    if TRAIN_DIR.is_symlink() or TRAIN_DIR.is_file():
        TRAIN_DIR.unlink()
    else:
        shutil.rmtree(TRAIN_DIR)

(TRAIN_DIR / "video").mkdir(parents=True, exist_ok=True)
(TRAIN_DIR / "gt").mkdir(parents=True, exist_ok=True)

def _load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f, strict=False)

def resolve_video_and_gt(case_input_dir: Path, case_id: str):
    """
    Возвращает (video_path, gt_path, info_record)
    Пытается взять пути из info.json, иначе подбирает по расширению.
    Если gt не находится внутри input-папки, пробует соседнюю папку *_test_file_gt.
    """
    info_path = case_input_dir / "info.json"
    assert info_path.exists(), f"Нет info.json в {case_input_dir}"
    info = _load_json(info_path)
    if isinstance(info, dict):
        info = [info]
    assert len(info) > 0, f"Пустой info.json в {case_input_dir}"
    rec = info[0]

    # В ноуте дальше используется ключ 'scene_change'
    src_rel = rec.get("source")
    sc_rel = rec.get("scene_change") or rec.get("gt")

    # video
    video_path = (case_input_dir / src_rel) if src_rel else None
    if (video_path is None) or (not video_path.exists()):
        videos = sorted((case_input_dir / "video").glob("*.mp4"))
        assert videos, f"Не нашёл mp4 в {case_input_dir/'video'}"
        video_path = videos[0]

    # gt json
    gt_path = (case_input_dir / sc_rel) if sc_rel else None
    if (gt_path is None) or (not gt_path.exists()):
        # 1) пробуем gt внутри input
        gts = sorted((case_input_dir / "gt").glob("*.json"))
        if gts:
            gt_path = gts[0]
        else:
            # 2) пробуем соседнюю папку *_test_file_gt (если вдруг ответы лежат там)
            sibling_gt = case_input_dir.parent / f"{case_id}_test_file_gt"
            if sibling_gt.exists():
                gts2 = sorted(sibling_gt.rglob("*.json"))
                assert gts2, f"Есть {sibling_gt}, но json внутри не найден"
                gt_path = gts2[0]
            else:
                raise FileNotFoundError(f"Не нашёл gt json ни в {case_input_dir/'gt'}, ни в sibling *_test_file_gt")

    return video_path, gt_path, rec

merged_info = []

for case_dir in case_dirs:
    case_id = case_dir.name.split("_")[0]  # "03" из "03_test_file_input"

    video_src, gt_src, rec = resolve_video_and_gt(case_dir, case_id)

    # создаём имена как в твоих ячейках: train_dataset/video/03.mp4 и train_dataset/gt/03.json
    video_dst = TRAIN_DIR / "video" / f"{case_id}.mp4"
    gt_dst = TRAIN_DIR / "gt" / f"{case_id}.json"

    # делаем symlink (быстро и без копирования)
    if video_dst.exists() or video_dst.is_symlink():
        video_dst.unlink()
    if gt_dst.exists() or gt_dst.is_symlink():
        gt_dst.unlink()

    video_dst.symlink_to(video_src)
    gt_dst.symlink_to(gt_src)

    # формируем запись для общего info.json
    new_rec = dict(rec)
    new_rec["source"] = f"video/{case_id}.mp4"
    new_rec["scene_change"] = f"gt/{case_id}.json"
    merged_info.append(new_rec)

# сохраняем общий info.json
with open(TRAIN_DIR / "info.json", "w", encoding="utf-8") as f:
    json.dump(merged_info, f, ensure_ascii=False, indent=2)

print("✅ Собран единый датасет:", TRAIN_DIR)
print("   Видео:", len(list((TRAIN_DIR/'video').glob('*.mp4'))))
print("   GT:", len(list((TRAIN_DIR/'gt').glob('*.json'))))
print("   Пример файлов video:", sorted([p.name for p in (TRAIN_DIR/'video').glob('*.mp4')])[:10])
print("   Пример файлов gt:", sorted([p.name for p in (TRAIN_DIR/'gt').glob('*.json')])[:10])

%cd /content

Mounted at /content/drive
📦 Не вижу распакованных *_test_file_input — распаковываю zip...
✅ Найдены кейсы: ['00_test_file_input', '01_test_file_input', '02_test_file_input', '03_test_file_input', '04_test_file_input']
✅ Собран единый датасет: /content/train_dataset
   Видео: 5
   GT: 5
   Пример файлов video: ['00.mp4', '01.mp4', '02.mp4', '03.mp4', '04.mp4']
   Пример файлов gt: ['00.json', '01.json', '02.json', '03.json', '04.json']
/content


In [ ]:
import os
print(os.path.exists("train_dataset/video/03.mp4"), "train_dataset/video/03.mp4")
print(os.path.exists("train_dataset/gt/03.json"), "train_dataset/gt/03.json")

True train_dataset/video/03.mp4
True train_dataset/gt/03.json


In [ ]:
import cv2

def read_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames

In [ ]:
import os, json

info_path = "/content/train_dataset/info.json"
with open(info_path, "r") as f:
    info = json.load(f, strict=False)

print("Видео в info.json:", len(info))
print("Ключи первой записи:", list(info[0].keys()))

# Пытаемся прочитать первое видео
src = info[0]["source"]  # обычно что-то вроде "video/03.mp4" или "video/....mp4"
video_path = os.path.join("/content/train_dataset", src)
frames = read_video(video_path)

print("Видео:", video_path)
print("Кадров:", len(frames), "shape[0]:", frames[0].shape)

Видео в info.json: 5
Ключи первой записи: ['source', 'scene_change', 'len']
Видео: /content/train_dataset/video/00.mp4
Кадров: 3250 shape[0]: (720, 1280, 3)


In [ ]:
import numpy as np
import cv2 # Для установки opencv воспользуйтесь командой в терминале conda install -c conda-forge opencv
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import os

%matplotlib inline

In [ ]:
import json
def load_json_from_file(filename):
    with open(filename, "r") as f:
        return json.load(f, strict=False)


def dump_json_to_file(obj, filename, **kwargs):
    with open(filename, "w") as f:
        json.dump(obj, f, **kwargs)

In [ ]:
video_dataset = load_json_from_file('train_dataset/info.json')
video_dataset

[{'source': 'video/00.mp4', 'scene_change': 'gt/00.json', 'len': 3250},
 {'source': 'video/01.mp4', 'scene_change': 'gt/01.json', 'len': 3392},
 {'source': 'video/02.mp4', 'scene_change': 'gt/02.json', 'len': 5662},
 {'source': 'video/03.mp4', 'scene_change': 'gt/03.json', 'len': 3396},
 {'source': 'video/04.mp4', 'scene_change': 'gt/04.json', 'len': 2326}]

In [ ]:
def read_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while(cap.isOpened()):
        ret, frame = cap.read()
        if ret==False:
            break
        yield frame
    cap.release()

In [ ]:
#Эти три клетки кода править не нужно
def calculate_matrix_dissolve(true_scd, predicted_scd, scene_len):
    predicted_scd = set(predicted_scd)
    tp, fp, tn, fn = 0, 0, 0, 0
    scene_len = scene_len
    checked_dissolve_segments = set()
    total_scene_dissolve_len = np.sum([dissolve_segment[1] - dissolve_segment[0] + 1 for dissolve_segment in true_scd])
    for scd in predicted_scd:
        for dissolve_segment in true_scd:
            if scd in range(dissolve_segment[0], dissolve_segment[1] + 1):
                if tuple(dissolve_segment) not in checked_dissolve_segments:
                    tp += 1
                    checked_dissolve_segments.add(tuple(dissolve_segment))
                break
        else:
            fp += 1
    fn = len(true_scd) - len(checked_dissolve_segments)
    tn = scene_len - total_scene_dissolve_len + len(true_scd) - tp - fp - fn
    return tp, fp, tn, fn

In [ ]:
def calculate_precision(tp, fp, tn, fn):
    return tp / max(1, (tp + fp))

In [ ]:
def calculate_recall(tp, fp, tn, fn):
    return tp / max(1, (tp + fn))

In [ ]:
def f1_score_dissolve(true_scd, predicted_scd, scene_len):
    tp, fp, tn, fn = calculate_matrix_dissolve(true_scd, predicted_scd, scene_len)
    precision_score = calculate_precision(tp, fp, tn, fn)
    recall_score = calculate_recall(tp, fp, tn, fn)
    if precision_score + recall_score == 0:
        return 0
    else:
        return 2 * precision_score * recall_score / (precision_score + recall_score)

In [ ]:
def f1_score_matrix(tp, fp, tn, fn):
    precision_score = calculate_precision(tp, fp, tn, fn)
    recall_score = calculate_recall(tp, fp, tn, fn)
    if precision_score + recall_score == 0:
        return 0
    else:
        return 2 * precision_score * recall_score / (precision_score + recall_score)

In [ ]:
def run_scene_change_detector_all_video_dissolve(scene_change_detector, dataset_path):
    video_dataset = load_json_from_file(os.path.join(dataset_path, 'info.json'))
    param_log = {
        '_mean_f1_score': []
    }
    for video_info in tqdm(video_dataset, leave=False):
        frames = read_video(os.path.join(dataset_path, video_info['source']))
        video_len = video_info['len']
        true_scene_changes = load_json_from_file(os.path.join(dataset_path, video_info['scene_change']))

        predicted_scene_changes, _, _ = scene_change_detector(frames)
        param_log['f1_score_{}'.format(video_info['source'])] = f1_score_dissolve(
            true_scene_changes.get('dissolve', []),
            predicted_scene_changes,
            video_len
        )
        video_tp, video_fp, video_tn, video_fn = calculate_matrix_dissolve(
            true_scene_changes.get('dissolve', []),
            predicted_scene_changes,
            video_len
        )
        param_log['tp_{}'.format(video_info['source'])] = video_tp
        param_log['fp_{}'.format(video_info['source'])] = video_fp
        param_log['tn_{}'.format(video_info['source'])] = video_tn
        param_log['fn_{}'.format(video_info['source'])] = video_fn
        param_log['_mean_f1_score'].append(param_log['f1_score_{}'.format(video_info['source'])])
    param_log['_mean_f1_score'] = np.mean(param_log['_mean_f1_score'])
    return param_log

In [ ]:
# GRADED CELL: scene_change_detector_dissolve
def scene_change_detector_dissolve(frames, with_vis=False):
    import numpy as np
    import cv2
    variant = 10  # baseline best solution + richer self-similarity matrix verifier

    # ---------------------------- helpers ----------------------------
    def _to_uint8(img):
        if img is None:
            return None
        if img.dtype == np.uint8:
            return img
        mx = float(np.max(img)) if img.size else 0.0
        if mx <= 1.5:
            img = img * 255.0
        return np.clip(img, 0, 255).astype(np.uint8)

    def _hist_1d_uint8(vals_u8, bins):
        idx = (vals_u8.astype(np.uint16) * bins) >> 8
        h = np.bincount(idx.ravel(), minlength=bins).astype(np.float32)
        s = float(h.sum()) + 1e-9
        return h / s

    def _l1_norm01(h1, h0):
        return float(np.sum(np.abs(h1 - h0))) * 0.5

    def _smooth_ma(x, k):
        x = np.asarray(x, dtype=np.float32)
        if k <= 1:
            return x
        if k % 2 == 0:
            k += 1
        pad = k // 2
        xp = np.pad(x, (pad, pad), mode="edge")
        w = np.ones(k, dtype=np.float32) / k
        return np.convolve(xp, w, mode="valid").astype(np.float32)

    def _robust_norm(x, p_lo=5, p_hi=95):
        x = np.asarray(x, dtype=np.float32)
        if x.size == 0:
            return x.astype(np.float32)
        core = x[1:] if len(x) > 1 else x
        lo = float(np.percentile(core, p_lo))
        hi = float(np.percentile(core, p_hi))
        if hi - lo < 1e-9:
            return np.zeros_like(x, dtype=np.float32)
        y = (x - lo) / (hi - lo)
        return np.clip(y, 0.0, 1.0).astype(np.float32)

    def _grad_mag_u8(y_u8):
        y = y_u8.astype(np.float32) / 255.0
        y = cv2.GaussianBlur(y, (3, 3), 0)
        gx = cv2.Sobel(y, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(y, cv2.CV_32F, 0, 1, ksize=3)
        g = np.abs(gx) + np.abs(gy)
        g = np.clip(g, 0.0, 3.0)
        return (g * (255.0 / 3.0)).astype(np.uint8)

    def _cosine_corr_u8(a_u8, b_u8):
        a = a_u8.astype(np.float32).ravel()
        b = b_u8.astype(np.float32).ravel()
        na = float(np.linalg.norm(a))
        nb = float(np.linalg.norm(b))
        if na * nb < 1e-9:
            return 0.0
        return float(np.dot(a, b) / (na * nb))

    def _warp_shift(img_u8, dx, dy):
        M = np.array([[1.0, 0.0, float(dx)], [0.0, 1.0, float(dy)]], dtype=np.float32)
        return cv2.warpAffine(img_u8, M, (img_u8.shape[1], img_u8.shape[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

    def _line_rmse_norm(vals, span_eps=1e-3):
        vals = np.asarray(vals, dtype=np.float64)
        if vals.size < 3:
            return 0.0
        span = float(np.max(vals) - np.min(vals))
        if span < 1e-9:
            return 0.0
        x = np.linspace(-1.0, 1.0, vals.size, dtype=np.float64)
        p = np.polyfit(x, vals, 1)
        pred = p[0] * x + p[1]
        rmse = float(np.sqrt(float(np.mean((vals - pred) ** 2))))
        return float(rmse / max(span, span_eps))

    def _extract_shape_metrics(seg):
        seg = np.asarray(seg, dtype=np.float32)
        if seg.size < 3:
            return {
                "width50": int(seg.size),
                "peak_ratio": 1.0,
                "zc2": 0,
                "rough": 0.0,
                "sym": 1.0,
            }
        pk = float(np.max(seg))
        meanv = float(np.mean(seg)) + 1e-6
        pk_i = int(np.argmax(seg))
        width50 = int(np.sum(seg >= 0.5 * pk))
        peak_ratio = float(pk / meanv)
        d2 = np.diff(seg, n=2)
        s = np.sign(d2)
        zc2 = int(np.sum(s[1:] * s[:-1] < 0)) if s.size >= 2 else 0
        rough = float(np.mean(np.abs(d2))) if d2.size else 0.0
        left = max(pk_i, 1)
        right = max(seg.size - pk_i - 1, 1)
        sym = float(min(left, right) / max(left, right))
        return {
            "width50": width50,
            "peak_ratio": peak_ratio,
            "zc2": zc2,
            "rough": rough,
            "sym": sym,
        }


    def _safe_inner_frame(st, en, alphas, proposal_seg, boundary_conf):
        L = en - st + 1
        if L <= 2:
            return int(st + np.argmin(np.abs(alphas - 0.5)))
        margin = 1 + int(round((1.0 - boundary_conf) * max(1.0, 0.18 * L)))
        lo = min(en, st + margin)
        hi = max(lo, en - margin)
        alpha_mid = int(st + np.argmin(np.abs(alphas - 0.5)))
        if proposal_seg.size:
            idxs = np.arange(st, en + 1, dtype=np.float64)
            w = proposal_seg.astype(np.float64)
            sw = float(np.sum(w))
            com = int(round(float(np.sum(idxs * w) / sw))) if sw > 1e-9 else int(round(0.5 * (st + en)))
        else:
            com = int(round(0.5 * (st + en)))
        safe_mid = int(round(0.5 * (lo + hi)))
        best = int(round(0.50 * alpha_mid + 0.25 * com + 0.25 * safe_mid))
        return int(np.clip(best, lo, hi))

    def _proposal_shape_ok(seg):
        m = _extract_shape_metrics(seg)
        L = len(seg)
        cond1 = m["width50"] >= max(2, int(0.16 * L))
        cond2 = m["peak_ratio"] <= 2.65
        cond3 = m["zc2"] <= max(4, L // 4)
        cond4 = m["rough"] <= 0.09
        cond5 = m["sym"] >= 0.22 or L >= 18
        return bool(cond1 and cond2 and cond3 and cond4 and cond5)

    def _alpha_fit_basic(Y_list, a_idx, b_idx, st, en):
        A = Y_list[a_idx].astype(np.float32)
        B = Y_list[b_idx].astype(np.float32)
        D = B - A
        denom = float(np.sum(D * D) + 1e-6)

        L = en - st + 1
        alphas = np.zeros(L, dtype=np.float32)
        errs = np.zeros(L, dtype=np.float32)
        A_flat = A.reshape(-1)
        D_flat = D.reshape(-1)

        for t, k in enumerate(range(st, en + 1)):
            I = Y_list[k].astype(np.float32).reshape(-1)
            num = float(np.dot(I - A_flat, D_flat))
            a = float(np.clip(num / denom, 0.0, 1.0))
            alphas[t] = a
            R = A_flat + a * D_flat
            errs[t] = float(np.mean(np.abs(I - R)) / 255.0)

        alphas_sm = _smooth_ma(alphas, k=5) if L >= 5 else alphas.copy()
        a_min = float(np.min(alphas_sm))
        a_max = float(np.max(alphas_sm))
        span = float(a_max - a_min)
        dir_sign = 1.0 if (alphas_sm[-1] - alphas_sm[0]) >= 0 else -1.0
        da = np.diff(alphas_sm)
        mono_score = float(np.mean(da * dir_sign >= -0.02)) if da.size else 1.0
        lin_rmse_norm = _line_rmse_norm(alphas_sm, 1e-3) if span > 1e-4 else 0.0
        return (
            alphas_sm,
            float(np.mean(errs)),
            float(np.percentile(errs, 90)),
            span,
            mono_score,
            lin_rmse_norm,
        )

    def _alpha_fit_multi(Y_list, Gs, Cs, a_idx, b_idx, st, en):
        Ay = Y_list[a_idx].astype(np.float32).reshape(-1) / 255.0
        By = Y_list[b_idx].astype(np.float32).reshape(-1) / 255.0
        Ag = Gs[a_idx].astype(np.float32).reshape(-1) / 255.0
        Bg = Gs[b_idx].astype(np.float32).reshape(-1) / 255.0
        Ac = Cs[a_idx].astype(np.float32).reshape(-1) / 255.0
        Bc = Cs[b_idx].astype(np.float32).reshape(-1) / 255.0

        Dy = By - Ay
        Dg = Bg - Ag
        Dc = Bc - Ac
        D = np.concatenate([Dy, 0.60 * Dg, 0.40 * Dc], axis=0)
        A = np.concatenate([Ay, 0.60 * Ag, 0.40 * Ac], axis=0)
        denom = float(np.dot(D, D) + 1e-6)

        L = en - st + 1
        alphas = np.zeros(L, dtype=np.float32)
        errs = np.zeros(L, dtype=np.float32)

        for t, k in enumerate(range(st, en + 1)):
            Iy = Y_list[k].astype(np.float32).reshape(-1) / 255.0
            Ig = Gs[k].astype(np.float32).reshape(-1) / 255.0
            Ic = Cs[k].astype(np.float32).reshape(-1) / 255.0
            I = np.concatenate([Iy, 0.60 * Ig, 0.40 * Ic], axis=0)
            num = float(np.dot(I - A, D))
            a = float(np.clip(num / denom, 0.0, 1.0))
            alphas[t] = a
            Ry = Ay + a * Dy
            errs[t] = float(np.mean(np.abs(Iy - Ry)))

        alphas_sm = _smooth_ma(alphas, k=5) if L >= 5 else alphas.copy()
        span = float(np.max(alphas_sm) - np.min(alphas_sm))
        dir_sign = 1.0 if (alphas_sm[-1] - alphas_sm[0]) >= 0 else -1.0
        da = np.diff(alphas_sm)
        mono_score = float(np.mean(da * dir_sign >= -0.02)) if da.size else 1.0
        lin_rmse_norm = _line_rmse_norm(alphas_sm, 1e-3) if span > 1e-4 else 0.0
        return (
            alphas_sm,
            float(np.mean(errs)),
            float(np.percentile(errs, 90)),
            span,
            mono_score,
            lin_rmse_norm,
        )



    def _self_similarity_bridge(Y_list, st, en):
        n_frames = len(Y_list)
        left_idxs = list(range(max(0, st - 4), st))
        right_idxs = list(range(en + 1, min(n_frames, en + 5)))
        if len(left_idxs) < 2 or len(right_idxs) < 2:
            return 1.0, 1.0, 1.0, 0.0

        left_refs = [cv2.resize(Y_list[i], (32, 18), interpolation=cv2.INTER_AREA) for i in left_idxs]
        right_refs = [cv2.resize(Y_list[i], (32, 18), interpolation=cv2.INTER_AREA) for i in right_idxs]

        def _mean_pair_corr(frames_small):
            vals = []
            for i in range(len(frames_small)):
                for j in range(i + 1, len(frames_small)):
                    vals.append(_cosine_corr_u8(frames_small[i], frames_small[j]))
            return float(np.mean(vals)) if vals else 1.0

        left_stab = _mean_pair_corr(left_refs)
        right_stab = _mean_pair_corr(right_refs)

        left_ref = left_refs[-1]
        right_ref = right_refs[0]
        endpoint_corr = _cosine_corr_u8(left_ref, right_ref)

        lc = []
        rc = []
        for k in range(st, en + 1):
            Ik = cv2.resize(Y_list[k], (32, 18), interpolation=cv2.INTER_AREA)
            lc.append(_cosine_corr_u8(Ik, left_ref))
            rc.append(_cosine_corr_u8(Ik, right_ref))
        lc = np.asarray(lc, dtype=np.float32)
        rc = np.asarray(rc, dtype=np.float32)
        left_desc = float(np.mean(np.diff(lc) <= 0.02)) if lc.size >= 2 else 1.0
        right_asc = float(np.mean(np.diff(rc) >= -0.02)) if rc.size >= 2 else 1.0
        bridge_mono = 0.5 * (left_desc + right_asc)
        cross_gap = float(np.clip(0.5 * ((left_stab + right_stab) - 2.0 * endpoint_corr), 0.0, 1.0))
        return left_stab, right_stab, bridge_mono, cross_gap


    def _self_similarity_matrix_features(Y_list, st, en):
        n_frames = len(Y_list)
        left_idxs = list(range(max(0, st - 4), st))
        mid_idxs = list(range(st, en + 1))
        right_idxs = list(range(en + 1, min(n_frames, en + 5)))
        if len(left_idxs) < 2 or len(right_idxs) < 2 or len(mid_idxs) < 3:
            return 1.0, 1.0, 1.0, 1.0, 0.0, 0.0

        idxs = left_idxs + mid_idxs + right_idxs
        small = [cv2.resize(Y_list[i], (24, 14), interpolation=cv2.INTER_AREA) for i in idxs]
        m = len(small)
        S = np.eye(m, dtype=np.float32)
        for i in range(m):
            for j in range(i + 1, m):
                c = _cosine_corr_u8(small[i], small[j])
                S[i, j] = c
                S[j, i] = c

        nL = len(left_idxs)
        nM = len(mid_idxs)
        nR = len(right_idxs)
        left_block = S[:nL, :nL]
        right_block = S[nL + nM:, nL + nM:]
        cross_block = S[:nL, nL + nM:]

        def _offdiag_mean(A):
            if A.shape[0] < 2:
                return 1.0
            mask = ~np.eye(A.shape[0], dtype=bool)
            vals = A[mask]
            return float(np.mean(vals)) if vals.size else 1.0

        past_self = _offdiag_mean(left_block)
        future_self = _offdiag_mean(right_block)
        cross_sim = float(np.mean(cross_block)) if cross_block.size else 1.0

        left_ref = S[nL - 1, nL:nL + nM]
        right_ref = S[nL + nM, nL:nL + nM]
        left_desc = float(np.mean(np.diff(left_ref) <= 0.02)) if left_ref.size >= 2 else 1.0
        right_asc = float(np.mean(np.diff(right_ref) >= -0.02)) if right_ref.size >= 2 else 1.0
        bridge = 0.5 * (left_desc + right_asc)

        band = []
        for d in range(1, min(4, m)):
            vals = np.diag(S, k=d)
            if vals.size:
                band.append(vals)
        if band:
            band = np.concatenate(band)
            diag_smooth = float(np.clip(1.0 - np.mean(np.abs(np.diff(band))) / 0.10, 0.0, 1.0)) if band.size >= 2 else 1.0
        else:
            diag_smooth = 1.0

        gap = float(np.clip(0.5 * ((past_self + future_self) - 2.0 * cross_sim), 0.0, 1.0))
        support = float(np.clip(0.32 * gap + 0.26 * bridge + 0.22 * diag_smooth + 0.10 * past_self + 0.10 * future_self, 0.0, 1.0))
        return past_self, future_self, cross_sim, diag_smooth, gap, support
    def _find_endpoints_local(Y_list, y_hist_list, c_hist_list, st, en, n_frames):
        best_a = max(0, st - 1)
        best_b = min(n_frames - 1, en + 1)
        best_score = -1e9
        a_lo = max(0, st - 4)
        a_hi = min(n_frames - 2, st + 2)
        b_lo0 = max(1, en - 2)
        b_hi = min(n_frames - 1, en + 4)
        for a in range(a_lo, a_hi + 1):
            b_lo = max(a + 1, b_lo0)
            for b in range(b_lo, b_hi + 1):
                if (b - a) < max(4, (en - st + 1) // 2):
                    continue
                alphas, err_mean, _err_p90, span, mono, lin_rmse = _alpha_fit_basic(Y_list, a, b, st, en)
                y_end = _l1_norm01(y_hist_list[b], y_hist_list[a])
                c_end = _l1_norm01(c_hist_list[b], c_hist_list[a])
                score = 1.20 * span + 0.90 * mono + 0.28 * (y_end + c_end) - 0.80 * err_mean - 0.60 * lin_rmse
                if score > best_score:
                    best_score = score
                    best_a, best_b = a, b
        return best_a, best_b

    def _flash_mask(bright_arr, chrom_arr, edge_arr, tiny_corr_arr):
        nloc = len(bright_arr)
        mask = np.zeros(nloc, dtype=np.float32)
        if nloc < 4:
            return mask
        bN_local = _robust_norm(bright_arr, 5, 95)
        eN_local = _robust_norm(edge_arr, 5, 95)
        cN_local = _robust_norm(chrom_arr, 5, 95)
        for i in range(1, nloc - 1):
            revert = False
            for d in (1, 2, 3):
                if i + d < nloc:
                    if abs(float(bright_arr[i + d]) - float(bright_arr[max(1, i - 1)])) < 0.018:
                        revert = True
                        break
            if (bN_local[i] > 0.82) and (cN_local[i] < 0.35) and (eN_local[i] < 0.45) and (tiny_corr_arr[i] > 0.970) and revert:
                mask[max(0, i - 1):min(nloc, i + 2)] = 1.0
        return _smooth_ma(mask, 3)

    def _stat_verifier(Y_list, st, en):
        mus = np.array([float(np.mean(Y_list[k])) / 255.0 for k in range(st, en + 1)], dtype=np.float32)
        vars_ = np.array([float(np.var(Y_list[k])) / (255.0 * 255.0) for k in range(st, en + 1)], dtype=np.float32)
        mu_rmse = _line_rmse_norm(_smooth_ma(mus, 5) if mus.size >= 5 else mus, 1e-3)
        var_sm = _smooth_ma(vars_, 5) if vars_.size >= 5 else vars_
        d2 = np.diff(var_sm, n=2)
        var_rough = float(np.mean(np.abs(d2))) if d2.size else 0.0
        var_zc = int(np.sum(np.sign(d2[1:]) * np.sign(d2[:-1]) < 0)) if d2.size >= 2 else 0
        return mu_rmse, var_rough, var_zc

    def _residual_structure(Y_list, a_idx, b_idx, st, en, alphas):
        A = Y_list[a_idx].astype(np.float32)
        B = Y_list[b_idx].astype(np.float32)
        D = B - A
        top1 = []
        top2 = []
        covs = []
        for off, k in enumerate(range(st, en + 1)):
            I = Y_list[k].astype(np.float32)
            a = float(alphas[off])
            R = A + a * D
            res = np.abs(I - R).astype(np.float32)
            blk = cv2.resize(res, (8, 5), interpolation=cv2.INTER_AREA).reshape(-1)
            s = float(np.sum(blk)) + 1e-6
            order = np.sort(blk)
            top1.append(float(order[-1] / s))
            top2.append(float((order[-1] + order[-2]) / s) if order.size >= 2 else float(order[-1] / s))
            covs.append(float(np.mean(blk > 0.40 * float(np.max(blk) + 1e-6))))
        return float(np.mean(top1)), float(np.mean(top2)), float(np.mean(covs))


    def _edge_enter_exit_ratio(curr_g_u8, prev_g_u8, thr=36):
        g0 = (prev_g_u8 >= thr).astype(np.uint8)
        g1 = (curr_g_u8 >= thr).astype(np.uint8)
        if int(g0.sum()) == 0 and int(g1.sum()) == 0:
            return 0.0, 1.0
        k = np.ones((3, 3), dtype=np.uint8)
        g0d = cv2.dilate(g0, k, iterations=1)
        g1d = cv2.dilate(g1, k, iterations=1)
        exit_ratio = float(np.sum((g0 > 0) & (g1d == 0)) / max(1, int(g0.sum())))
        enter_ratio = float(np.sum((g1 > 0) & (g0d == 0)) / max(1, int(g1.sum())))
        change = 0.5 * (exit_ratio + enter_ratio)
        balance = 1.0 - min(1.0, abs(exit_ratio - enter_ratio) / max(0.10, change + 1e-6))
        balance = float(np.clip(balance, 0.0, 1.0))
        return float(change), balance

    def _calc_hilo(signal, low_light=False, extra_relaxed=False):
        core = signal[1:] if len(signal) > 1 else signal
        med = float(np.median(core))
        mad = float(np.median(np.abs(core - med)) + 1e-9)
        p80 = float(np.percentile(core, 80))
        p90 = float(np.percentile(core, 90))
        hi_floor = 0.030 if (p90 < 0.030 and not low_light) else 0.035
        lo_floor = 0.017 if (p90 < 0.030 and not low_light) else 0.020
        hi = max(p90, med + 2.3 * mad, hi_floor)
        lo = max(p80, med + 1.05 * mad, lo_floor, 0.55 * hi)
        if extra_relaxed:
            hi *= 0.95
            lo *= 0.95
        return float(hi), float(lo)

    def _extract_proposals(signal, n_frames, min_len, max_len, hi, lo, use_shape=False):
        props = []
        i = 1
        while i < n_frames:
            if signal[i] >= hi:
                st = i
                while i < n_frames and signal[i] >= lo:
                    i += 1
                en = i - 1
                L = en - st + 1
                if not (min_len <= L <= max_len):
                    continue
                seg = signal[st:en + 1]
                pk = float(np.max(seg))
                pk_i = int(np.argmax(seg))
                left_min = float(np.min(seg[:max(1, pk_i + 1)]))
                right_min = float(np.min(seg[pk_i:]))
                hill_ok = (left_min < 0.90 * pk) and (right_min < 0.90 * pk)
                if not hill_ok and L < 14:
                    continue
                if use_shape and not _proposal_shape_ok(seg):
                    continue
                props.append((st, en))
            else:
                i += 1
        return props

    # ---------------------------- params ----------------------------
    W, H = 160, 90
    TINY_W, TINY_H = 48, 27
    Y_BINS = 32
    C_BINS = 16
    DIFF_THR = 18
    BX, BY = 10, 6
    BLOCK_THR = 10
    MIN_LEN, MAX_LEN = 7, 260
    MERGE_GAP = 2
    SPAN_MIN = 0.45
    MONO_MIN = 0.68
    LIN_RMSE_MAX = 0.30
    ERR_MEAN_MAX = 0.18
    ERR_P90_MAX = 0.26
    BLOCK_HARD = 0.55
    SHIFT_HARD = 0.12

    # ---------------------------- Stage 0: read frames + cheap cues ----------------------------
    buf = [] if with_vis else None
    Y = []
    tiny = []
    y_hist = []
    c_hist = []
    grad = []
    grad_small = []
    c_small = []
    meanY = []
    grad_energy = []
    chroma_spread = []

    yhist_l1 = [0.0]
    chrom_l1 = [0.0]
    edge_mad = [0.0]
    diff_frac = [0.0]
    block_cov = [0.0]
    bright = [0.0]
    shift_mag = [0.0]
    pc_resp = [0.0]
    ecr_change = [0.0]
    ecr_balance = [1.0]
    tiny_corr = [1.0]
    hann = None

    prev_y = None
    prev_yh = None
    prev_ch = None
    prev_g = None
    prev_mean = None
    prev_tiny = None

    for fr in frames:
        fr = _to_uint8(fr)
        if fr is None:
            continue
        if with_vis:
            buf.append(fr)
        if fr.ndim == 2:
            fr = cv2.cvtColor(fr, cv2.COLOR_GRAY2BGR)
        elif fr.shape[2] == 4:
            fr = fr[:, :, :3]

        fr = cv2.resize(fr, (W, H), interpolation=cv2.INTER_AREA)
        ycc = cv2.cvtColor(fr, cv2.COLOR_BGR2YCrCb)
        y = ycc[:, :, 0]
        Cr = ycc[:, :, 1]
        Cb = ycc[:, :, 2]

        Y.append(y)
        tiny_y = cv2.resize(y, (TINY_W, TINY_H), interpolation=cv2.INTER_AREA)
        tiny.append(tiny_y)
        mY = float(np.mean(y))
        meanY.append(mY)
        yh = _hist_1d_uint8(y, Y_BINS)
        hCr = _hist_1d_uint8(Cr, C_BINS)
        hCb = _hist_1d_uint8(Cb, C_BINS)
        ch = np.concatenate([hCr, hCb], axis=0)
        y_hist.append(yh)
        c_hist.append(ch)
        g = _grad_mag_u8(y)
        grad.append(g)
        grad_small.append(cv2.resize(g, (TINY_W, TINY_H), interpolation=cv2.INTER_AREA))
        c_small.append(cv2.resize(np.dstack([Cr, Cb]), (TINY_W, TINY_H), interpolation=cv2.INTER_AREA))
        grad_energy.append(float(np.mean(g)) / 255.0)
        chroma_spread.append((float(np.std(Cr)) + float(np.std(Cb))) / (2.0 * 128.0))

        if prev_y is None:
            prev_y = y
            prev_yh = yh
            prev_ch = ch
            prev_g = g
            prev_mean = mY
            prev_tiny = tiny_y
            continue

        if hann is None:
            try:
                hann = cv2.createHanningWindow((W, H), cv2.CV_32F)
            except Exception:
                hann = None

        dx = dy2 = resp = 0.0
        if hann is not None:
            (dx, dy2), resp = cv2.phaseCorrelate(prev_y.astype(np.float32), y.astype(np.float32), hann)
            if abs(dx) > 0.25 * W or abs(dy2) > 0.25 * H:
                dx, dy2, resp = 0.0, 0.0, 0.0
        shift_mag.append(float(np.hypot(dx, dy2) / max(W, H)))
        pc_resp.append(float(resp))

        y_for_diff_prev = prev_y
        g_for_diff_prev = prev_g
        tiny_for_corr_prev = prev_tiny
        if (abs(dx) > 1e-4 or abs(dy2) > 1e-4) and (resp > 0.02):
            y_for_diff_prev = _warp_shift(prev_y, dx, dy2)
            g_for_diff_prev = _warp_shift(prev_g, dx, dy2)
            tiny_for_corr_prev = _warp_shift(prev_tiny, dx * (TINY_W / float(W)), dy2 * (TINY_H / float(H)))

        yhist_l1.append(_l1_norm01(yh, prev_yh))
        chrom_l1.append(_l1_norm01(ch, prev_ch))
        edge_mad.append(float(np.mean(cv2.absdiff(g, g_for_diff_prev))) / 255.0)
        dy = cv2.absdiff(y, y_for_diff_prev)
        diff_frac.append(float(np.mean(dy > DIFF_THR)))
        blk = cv2.resize(dy, (BX, BY), interpolation=cv2.INTER_AREA)
        block_cov.append(float(np.mean(blk > BLOCK_THR)))
        bright.append(abs(mY - prev_mean) / 255.0)
        tiny_corr.append(_cosine_corr_u8(tiny_y, tiny_for_corr_prev))
        ecr_ch, ecr_bal = _edge_enter_exit_ratio(g, g_for_diff_prev)
        ecr_change.append(ecr_ch)
        ecr_balance.append(ecr_bal)

        prev_y = y
        prev_yh = yh
        prev_ch = ch
        prev_g = g
        prev_mean = mY
        prev_tiny = tiny_y

    n = len(Y)
    if n < 2:
        return [], [], ([0.0] if n == 1 else [])

    yhist_l1 = np.asarray(yhist_l1[:n], dtype=np.float32)
    chrom_l1 = np.asarray(chrom_l1[:n], dtype=np.float32)
    edge_mad = np.asarray(edge_mad[:n], dtype=np.float32)
    diff_frac = np.asarray(diff_frac[:n], dtype=np.float32)
    block_cov = np.asarray(block_cov[:n], dtype=np.float32)
    bright = np.asarray(bright[:n], dtype=np.float32)
    shift_mag = np.asarray((shift_mag + [0.0])[:n], dtype=np.float32)
    pc_resp = np.asarray((pc_resp + [0.0])[:n], dtype=np.float32)
    ecr_change = np.asarray((ecr_change + [0.0])[:n], dtype=np.float32)
    ecr_balance = np.asarray((ecr_balance + [1.0])[:n], dtype=np.float32)
    tiny_corr = np.asarray((tiny_corr + [1.0])[:n], dtype=np.float32)
    grad_energy = np.asarray(grad_energy[:n], dtype=np.float32)
    chroma_spread = np.asarray(chroma_spread[:n], dtype=np.float32)

    yhist_lag2 = np.zeros(n, dtype=np.float32)
    chrom_lag2 = np.zeros(n, dtype=np.float32)
    yhist_lag6 = np.zeros(n, dtype=np.float32)
    chrom_lag6 = np.zeros(n, dtype=np.float32)
    for i in range(2, n):
        yhist_lag2[i] = _l1_norm01(y_hist[i], y_hist[i - 2])
        chrom_lag2[i] = _l1_norm01(c_hist[i], c_hist[i - 2])
    for i in range(6, n):
        yhist_lag6[i] = _l1_norm01(y_hist[i], y_hist[i - 6])
        chrom_lag6[i] = _l1_norm01(c_hist[i], c_hist[i - 6])

    avg_y = float(np.mean(meanY)) if meanY else 128.0
    low_light = avg_y < 70.0

    # ---------------------------- Stage 0: proposal signal ----------------------------
    yN = _robust_norm(yhist_l1, 5, 95)
    cN = _robust_norm(chrom_l1, 5, 95)
    eN = _robust_norm(edge_mad, 5, 95)
    dN = _robust_norm(diff_frac, 5, 95)
    bN = _robust_norm(bright, 5, 95)
    bcN = _robust_norm(block_cov, 5, 95)
    shN = _robust_norm(shift_mag, 5, 95)
    ecrN = _robust_norm(ecr_change, 5, 95)
    ecrBalN = _robust_norm(ecr_balance, 5, 95)

    w_y, w_c, w_e, w_d, w_b = 0.30, 0.28, 0.22, 0.12, 0.08
    extra_relaxed = False
    if variant == 5:
        high_motion = (float(np.mean(block_cov[1:])) > 0.18) or (float(np.mean(shift_mag[1:])) > 0.035)
        low_texture = float(np.mean(grad_energy)) < 0.16
        low_chroma = float(np.mean(chroma_spread)) < 0.10
        if low_light:
            w_y += 0.06
            w_e += 0.06
            w_b += 0.06
            w_c -= 0.10
            w_d -= 0.08
        if high_motion:
            w_y += 0.06
            w_c += 0.06
            w_d -= 0.07
            w_b -= 0.03
            extra_relaxed = True
        if low_texture:
            w_e -= 0.08
            w_y += 0.04
            w_c += 0.02
        if low_chroma:
            w_c -= 0.08
            w_y += 0.05
            w_e += 0.03
    else:
        if low_light:
            w_y += 0.06
            w_e += 0.06
            w_b += 0.06
            w_c -= 0.10
            w_d -= 0.08

    ws = np.array([w_y, w_c, w_e, w_d, w_b], dtype=np.float32)
    ws = np.clip(ws, 0.0, None)
    ws = ws / (float(ws.sum()) + 1e-9)
    w_y, w_c, w_e, w_d, w_b = ws.tolist()

    lag2N = _robust_norm(0.65 * yhist_lag2 + 0.35 * chrom_lag2, 5, 95)
    lag6N = _robust_norm(0.65 * yhist_lag6 + 0.35 * chrom_lag6, 5, 95)

    motion_proxy = np.maximum(0.0, diff_frac - 0.5 * (yhist_l1 + chrom_l1)).astype(np.float32)
    mN = _robust_norm(motion_proxy, 5, 95)

    proposal_raw = (w_y * yN + w_c * cN + w_e * eN + w_d * dN + w_b * bN + 0.10 * lag2N + 0.05 * lag6N + 0.12 * ecrN * (0.55 + 0.45 * ecrBalN)).astype(np.float32)
    proposal_raw = np.maximum(0.0, proposal_raw - 0.16 * mN - 0.10 * bcN - 0.08 * shN).astype(np.float32)

    if variant == 3:
        flashN = _flash_mask(bright, chrom_l1, edge_mad, tiny_corr)
        proposal_raw = np.maximum(0.0, proposal_raw - 0.32 * flashN).astype(np.float32)

    if variant == 2:
        proposal_scales = [_smooth_ma(proposal_raw, k=5), _smooth_ma(proposal_raw, k=9), _smooth_ma(proposal_raw, k=15)]
        proposal_sm = np.max(np.stack(proposal_scales, axis=0), axis=0).astype(np.float32)
        proposals = []
        for signal in proposal_scales:
            hi, lo = _calc_hilo(signal, low_light=low_light, extra_relaxed=extra_relaxed)
            proposals.extend(_extract_proposals(signal, n, MIN_LEN, MAX_LEN, hi, lo, use_shape=False))
    else:
        proposal_sm = _smooth_ma(proposal_raw, k=9)
        hi, lo = _calc_hilo(proposal_sm, low_light=low_light, extra_relaxed=extra_relaxed)
        proposals = _extract_proposals(proposal_sm, n, MIN_LEN, MAX_LEN, hi, lo, use_shape=(variant == 4))

    if not proposals:
        return [], [], proposal_sm.tolist()

    proposals.sort(key=lambda x: x[0])
    merged = []
    for st, en in proposals:
        if not merged:
            merged.append([st, en])
        else:
            if st <= merged[-1][1] + MERGE_GAP:
                merged[-1][1] = max(merged[-1][1], en)
            else:
                merged.append([st, en])

    # ---------------------------- Stage 1: alpha-fit verification ----------------------------
    verified = []
    cut_like_df = max(float(np.percentile(diff_frac[1:], 99.7)), 0.60)
    cut_like_prop = max(float(np.percentile(proposal_raw[1:], 99.7)), 0.35)

    for st, en in merged:
        L = en - st + 1
        if L < MIN_LEN or L > MAX_LEN:
            continue
        if float(np.max(diff_frac[st:en + 1])) > cut_like_df:
            continue
        if float(np.max(proposal_raw[st:en + 1])) > cut_like_prop and L < 18:
            continue

        if variant == 1:
            a, b = _find_endpoints_local(Y, y_hist, c_hist, st, en, n)
        else:
            a = max(0, st - 1)
            b = min(n - 1, en + 1)
            if b <= a:
                a = st
                b = en
        if b <= a:
            continue

        y_end = _l1_norm01(y_hist[b], y_hist[a])
        c_end = _l1_norm01(c_hist[b], c_hist[a])
        corr_end = _cosine_corr_u8(tiny[a], tiny[b])
        mean_drift = abs(float(meanY[b]) - float(meanY[a])) / 255.0
        left_stab, right_stab, bridge_mono, cross_gap = _self_similarity_bridge(Y, st, en)
        ssm_left, ssm_right, ssm_cross, ssm_diag, ssm_gap, ssm_support = _self_similarity_matrix_features(Y, st, en)
        if (corr_end > 0.985) and (c_end < 0.05) and (y_end < 0.07) and (mean_drift > 0.05):
            continue
        if (bridge_mono < 0.58) and (cross_gap < 0.10):
            continue
        if (min(left_stab, right_stab) < 0.82) and (L >= 10):
            continue
        if (ssm_gap < 0.07) and (ssm_support < 0.42) and (L >= 9):
            continue
        if (ssm_diag < 0.36) and (ssm_gap < 0.10):
            continue

        bc = float(np.mean(block_cov[st:en + 1]))
        sh = float(np.mean(shift_mag[st:en + 1]))
        rp = float(np.mean(pc_resp[st:en + 1]))
        motion_hard = (bc > BLOCK_HARD) or (sh > SHIFT_HARD and rp > 0.05)

        if variant == 8:
            alphas, err_mean, err_p90, span, mono, lin_rmse = _alpha_fit_multi(Y, grad_small, c_small, a, b, st, en)
        else:
            alphas, err_mean, err_p90, span, mono, lin_rmse = _alpha_fit_basic(Y, a, b, st, en)

        span_min = SPAN_MIN + (0.10 if motion_hard else 0.0)
        mono_min = MONO_MIN + (0.10 if motion_hard else 0.0)
        lin_max = LIN_RMSE_MAX - (0.08 if motion_hard else 0.0)
        err_mean_max = ERR_MEAN_MAX - (0.04 if motion_hard else 0.0)
        err_p90_max = ERR_P90_MAX - (0.06 if motion_hard else 0.0)

        if variant == 10:
            if span < 0.25:
                continue
            if (err_mean > 0.28) or (err_p90 > 0.40):
                continue
            endpoint_sep = float(min(alphas[0], alphas[-1]) < 0.45 and max(alphas[0], alphas[-1]) > 0.55)
            low_end_guard = 1.0 if (y_end + c_end) >= 0.10 else 0.0
            soft_score = (
                0.28 * np.clip((span - 0.25) / 0.55, 0.0, 1.0)
                + 0.20 * np.clip((mono - 0.55) / 0.40, 0.0, 1.0)
                + 0.16 * np.clip((0.42 - lin_rmse) / 0.42, 0.0, 1.0)
                + 0.14 * np.clip((0.24 - err_mean) / 0.24, 0.0, 1.0)
                + 0.10 * np.clip((y_end + c_end) / 0.22, 0.0, 1.0)
                + 0.06 * endpoint_sep
                + 0.04 * low_end_guard
                + 0.08 * np.clip(float(np.mean(proposal_sm[st:en + 1])) / 0.12, 0.0, 1.0)
                - 0.07 * np.clip(bc / 0.8, 0.0, 1.0)
                - 0.05 * np.clip(sh / 0.2, 0.0, 1.0)
            )
        else:
            if span < span_min:
                continue
            if mono < mono_min:
                continue
            if lin_rmse > lin_max:
                continue
            if (err_mean > err_mean_max) or (err_p90 > err_p90_max):
                continue
            if not (min(alphas[0], alphas[-1]) < 0.45 and max(alphas[0], alphas[-1]) > 0.55):
                if not (span > 0.65 and mono > 0.82 and err_mean < (0.12 if motion_hard else 0.14)):
                    continue
            if (y_end + c_end) < 0.10:
                if not (span > 0.62 and mono > 0.80 and err_mean < (0.12 if motion_hard else 0.14)):
                    continue
            soft_score = None

        if variant == 7:
            mu_rmse, var_rough, var_zc = _stat_verifier(Y, st, en)
            if (mu_rmse > 0.70) or (var_rough > 0.010 and var_zc > max(4, L // 5)):
                continue

        if variant == 9:
            top1, top2, cov = _residual_structure(Y, a, b, st, en, alphas)
            if (top2 > 0.76) and (cov < 0.24) and (span < 0.72):
                continue

        boundary_conf = float(np.clip(
            0.40 * np.clip((y_end + c_end) / 0.22, 0.0, 1.0)
            + 0.35 * np.clip((span - 0.25) / 0.55, 0.0, 1.0)
            + 0.25 * np.clip((0.30 - lin_rmse) / 0.30, 0.0, 1.0),
            0.0,
            1.0,
        ))
        best = _safe_inner_frame(st, en, alphas, proposal_sm[st:en + 1], boundary_conf)
        if L >= 22:
            w = proposal_sm[st:en + 1].astype(np.float64)
            sw = float(np.sum(w))
            if sw > 1e-9:
                idxs = np.arange(st, en + 1, dtype=np.float64)
                com = int(np.round(float(np.sum(idxs * w) / sw)))
                if abs(com - best) > 6:
                    best = com

        seg_strength = float(np.mean(proposal_sm[st:en + 1])) + 0.10 * float(np.max(proposal_sm[st:en + 1]))
        alpha_strength = float(span + mono - 0.70 * err_mean - 0.50 * lin_rmse)
        content_strength = float(y_end + c_end)
        score = float(
            0.46 * alpha_strength
            + 0.22 * seg_strength
            + 0.22 * content_strength
            + 0.015 * L
            - 0.10 * bc
            - 0.06 * sh
        )
        if soft_score is not None:
            score = float(soft_score)
        score = float(
            score
            + 0.05 * bridge_mono
            + 0.04 * cross_gap
            + 0.02 * min(left_stab, right_stab)
            + 0.06 * ssm_support
            + 0.03 * ssm_gap
            + 0.02 * ssm_diag
        )

        verified.append((st, en, best, score))

    if not verified:
        return [], [], proposal_sm.tolist()

    verified.sort(key=lambda x: x[0])
    merged_v = []
    for st, en, best, sc in verified:
        if not merged_v:
            merged_v.append([st, en, best, sc])
        else:
            if st <= merged_v[-1][1] + 2:
                merged_v[-1][1] = max(merged_v[-1][1], en)
                if sc > merged_v[-1][3]:
                    merged_v[-1][2] = best
                    merged_v[-1][3] = sc
            else:
                merged_v.append([st, en, best, sc])

    scores = np.array([x[3] for x in merged_v], dtype=np.float32)
    if variant == 10:
        thr = float(max(np.percentile(scores, 55), 0.38))
    else:
        thr = float(max(np.percentile(scores, 50), 0.10))
    merged_v = [x for x in merged_v if x[3] >= thr]

    merged_v.sort(key=lambda x: x[3], reverse=True)
    maxK = min(25, max(8, n // 500))
    merged_v = merged_v[:maxK]
    merged_v.sort(key=lambda x: x[0])

    predicted = sorted(set(int(x[2]) for x in merged_v))
    visualization = []
    if with_vis and buf:
        vis_offset = 5
        for idx in predicted:
            if 0 <= idx < len(buf):
                s0 = max(0, idx - vis_offset)
                e0 = min(len(buf) - 1, idx + vis_offset)
                visualization.append([buf[s0], buf[idx], buf[e0]])

    return predicted, visualization, proposal_sm.tolist()

In [ ]:
video_dataset_path = 'train_dataset'

In [ ]:
run_scene_change_detector_all_video_dissolve(scene_change_detector_dissolve, video_dataset_path)

  0%|          | 0/5 [00:00<?, ?it/s]

{'_mean_f1_score': np.float64(0.49995004995005),
 'f1_score_video/00.mp4': 0.7692307692307693,
 'tp_video/00.mp4': 5,
 'fp_video/00.mp4': 0,
 'tn_video/00.mp4': np.int64(3195),
 'fn_video/00.mp4': 3,
 'f1_score_video/01.mp4': 0.25,
 'tp_video/01.mp4': 1,
 'fp_video/01.mp4': 6,
 'tn_video/01.mp4': np.int64(3382),
 'fn_video/01.mp4': 0,
 'f1_score_video/02.mp4': 0.9090909090909091,
 'tp_video/02.mp4': 5,
 'fp_video/02.mp4': 1,
 'tn_video/02.mp4': np.int64(5547),
 'fn_video/02.mp4': 0,
 'f1_score_video/03.mp4': 0,
 'tp_video/03.mp4': 0,
 'fp_video/03.mp4': 5,
 'tn_video/03.mp4': np.int64(3331),
 'fn_video/03.mp4': 1,
 'f1_score_video/04.mp4': 0.5714285714285715,
 'tp_video/04.mp4': 2,
 'fp_video/04.mp4': 2,
 'tn_video/04.mp4': np.int64(2284),
 'fn_video/04.mp4': 1}